# Data loaders

> Data loading functionalities

In [ ]:
#| default_exp data.load

In [ ]:
#| hide
from nbdev.showdoc import *
from nbdev.cli import *

In [ ]:
#| export
from fastai.vision.all import *
from fastai.data.all import *
from pathlib import Path
import pandas as pd

/Users/franckalbinet/mambaforge/envs/spanda/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loaders

In [ ]:
#| export
def get_spectra_files(path):
    "Return paths of spectra `.csv` files in path"
    return [f for f in path.ls() if re.match('\d', f.name)]

In [ ]:
#|eval: false
path = Path('../_data/kssl-mirs')
fnames = path.ls(); fnames

(#44101) [Path('../_data/kssl-mirs/180338'),Path('../_data/kssl-mirs/172221'),Path('../_data/kssl-mirs/177753'),Path('../_data/kssl-mirs/184798'),Path('../_data/kssl-mirs/53759'),Path('../_data/kssl-mirs/74947'),Path('../_data/kssl-mirs/176681'),Path('../_data/kssl-mirs/1855'),Path('../_data/kssl-mirs/175004'),Path('../_data/kssl-mirs/34499')...]

In [ ]:
#|eval: false
get_spectra_files(fnames[0])

[Path('../_data/kssl-mirs/180338/260043XS04.csv'),
 Path('../_data/kssl-mirs/180338/260043XS01.csv'),
 Path('../_data/kssl-mirs/180338/260043XS02.csv'),
 Path('../_data/kssl-mirs/180338/260043XS03.csv')]

In [ ]:
#| export
class SpectraSetup(Transform):
    "Transform that converts to numpy arrays"
    def __init__(self):
        pass
    def encodes(self, fnames):
        n = pd.read_csv(fnames[0]).shape[0]
        m = len(fnames)
        x = np.empty((m,n))
        for i, fname in enumerate(fnames):
            x[i,:] = pd.read_csv(fname)['absorbance'].values
        return tensor(x)

In [ ]:
#| export
def get_target(path, analytes=None):
    path_target = [f for f in path.ls() if re.match('target', f.name)][0]
    df = pd.read_csv(path_target)
    if analytes:
        df = df[df.analyte.isin(analytes)]
    return df['value'].values

In [ ]:
#|eval: false
df = get_target(fnames[0]); df

array([ 2.8077302, 22.033033 , 13.7791402])

In [ ]:
#| export
def SpectraBlock():
    return TransformBlock(type_tfms=SpectraSetup())

In [ ]:
#| export
def random_weighted_avg(x):
    "Transform spectra replicates taking their random weighted averages"
    n = len(x)
    def weights(n):
        weights = torch.rand(n)
        return (weights/weights.sum()).unsqueeze(dim=0)
    return torch.matmul(weights(n), x)

In [ ]:
#|eval: false
dblock = DataBlock(
    blocks=(SpectraBlock, RegressionBlock),
    get_x=get_spectra_files,
    get_y=partial(get_target, analytes=[725]),
    splitter=RandomSplitter(valid_pct=0.1, seed=42),
    item_tfms=[Transform(random_weighted_avg)]
)

In [ ]:
#|eval: false
dsets = dblock.datasets(fnames)
dsets.train[0]

(tensor([[0.2258, 0.2260, 0.2263,  ..., 1.5891, 1.5789, 1.5701],
         [0.2639, 0.2641, 0.2644,  ..., 1.6350, 1.6246, 1.6143],
         [0.2816, 0.2819, 0.2821,  ..., 1.5537, 1.5427, 1.5320],
         [0.3072, 0.3074, 0.3076,  ..., 1.6325, 1.6217, 1.6133]]),
 tensor([0.2244]))

In [ ]:
#|eval: false
dls = dblock.dataloaders(fnames, bs=16)

In [ ]:
#|eval: false
dls.train.one_batch()

(tensor([[[0.2472, 0.2475, 0.2477,  ..., 1.4216, 1.4140, 1.4086]],
 
         [[0.2291, 0.2297, 0.2303,  ..., 1.5253, 1.5212, 1.5170]],
 
         [[0.3690, 0.3695, 0.3702,  ..., 1.8249, 1.8175, 1.8108]],
 
         ...,
 
         [[0.2799, 0.2801, 0.2803,  ..., 1.5486, 1.5538, 1.5566]],
 
         [[0.1669, 0.1672, 0.1676,  ..., 1.3549, 1.3553, 1.3575]],
 
         [[0.5311, 0.5316, 0.5321,  ..., 2.0411, 2.0384, 2.0342]]]),
 tensor([[0.0000],
         [0.2071],
         [1.0397],
         [1.8629],
         [0.4369],
         [0.6804],
         [1.7309],
         [0.3431],
         [0.7035],
         [0.6687],
         [0.4171],
         [1.0808],
         [0.9890],
         [0.1135],
         [0.1327],
         [1.0052]]))

In [ ]:
#|eval: false
dls.valid.one_batch()

(tensor([[[0.3401, 0.3407, 0.3413,  ..., 1.6816, 1.6644, 1.6470]],
 
         [[1.0220, 1.0217, 1.0213,  ..., 1.3201, 1.3187, 1.3169]],
 
         [[0.2562, 0.2563, 0.2565,  ..., 1.5045, 1.5033, 1.5014]],
 
         ...,
 
         [[0.2171, 0.2173, 0.2175,  ..., 1.6235, 1.6197, 1.6158]],
 
         [[0.1625, 0.1629, 0.1633,  ..., 1.3806, 1.3718, 1.3651]],
 
         [[0.1358, 0.1358, 0.1358,  ..., 1.3529, 1.3410, 1.3296]]]),
 tensor([[0.4968],
         [0.3443],
         [0.0000],
         [0.0000],
         [0.9650],
         [0.6087],
         [1.3657],
         [0.0461],
         [1.0763],
         [1.4860],
         [0.7453],
         [0.0448],
         [2.7428],
         [0.1603],
         [0.6544],
         [0.5604]]))